# 🚀 LLM Simple - Compilation pour Godot 4.2+

**Objectif:** Créer un GDExtension Windows pour utiliser Qwen2.5-3B dans Godot

**Durée totale:** ~12-15 minutes

**Instructions:** Exécutez chaque cellule dans l'ordre avec Shift+Enter

In [1]:
# ==========================================
# CELLULE 1: Setup et Installation
# ==========================================
import os
import subprocess
from pathlib import Path

print("="*70)
print("🔧 CELLULE 1: SETUP ET INSTALLATION")
print("="*70)
print()

# Nettoyage complet
print("🧹 Nettoyage environnement...")
os.chdir("/content")
os.system("rm -rf godot-cpp llama.cpp llm_simple addons llama_build *.o *.a 2>/dev/null")

# Création structure
print("📁 Création structure dossiers...")
dirs = [
    "llm_simple/include",
    "llm_simple/src",
    "addons/llm_simple/bin",
    "llama_build"
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {d}")

print()

# Installation outils
print("📦 Installation outils de compilation...")
result = subprocess.run("apt-get update -qq", shell=True, capture_output=True)
if result.returncode != 0:
    print("⚠️ Warning: apt-get update failed")

result = subprocess.run("apt-get install -y -qq mingw-w64 zip", shell=True, capture_output=True)
if result.returncode != 0:
    raise Exception("❌ Installation mingw-w64 échouée")
print("  ✓ mingw-w64 installé")

result = subprocess.run("pip install -q scons", shell=True, capture_output=True)
if result.returncode != 0:
    raise Exception("❌ Installation scons échouée")
print("  ✓ scons installé")

print()

# Vérification compilateur
print("🔍 Vérification compilateur...")
result = subprocess.run("x86_64-w64-mingw32-g++ --version", shell=True, capture_output=True, text=True)
if result.returncode == 0:
    version = result.stdout.split('\n')[0]
    print(f"  ✓ {version}")
else:
    raise Exception("❌ Compilateur MinGW non trouvé")

print()
print("="*70)
print("✅ CELLULE 1 TERMINÉE")
print("="*70)

🔧 CELLULE 1: SETUP ET INSTALLATION

🧹 Nettoyage environnement...
📁 Création structure dossiers...
  ✓ llm_simple/include
  ✓ llm_simple/src
  ✓ addons/llm_simple/bin
  ✓ llama_build

📦 Installation outils de compilation...
  ✓ mingw-w64 installé
  ✓ scons installé

🔍 Vérification compilateur...
  ✓ x86_64-w64-mingw32-g++ (GCC) 10-win32 20220113

✅ CELLULE 1 TERMINÉE


In [2]:
# ==========================================
# CELLULE 2: Création des Fichiers Sources
# ==========================================
print("="*70)
print("📝 CELLULE 2: CRÉATION FICHIERS SOURCES")
print("="*70)
print()

os.chdir("/content")

# Header LLMSimple
print("📄 Création llm_simple.h...")
header_content = """#pragma once
#include <godot_cpp/classes/node.hpp>
#include <godot_cpp/core/class_db.hpp>

struct llama_model;
struct llama_context;
struct llama_sampler;

namespace godot {
class LLMSimple : public Node {
    GDCLASS(LLMSimple, Node)
private:
    llama_model* model;
    llama_context* ctx;
    llama_sampler* sampler;
    int n_ctx;
    bool is_loaded;
    String last_error;
protected:
    static void _bind_methods();
public:
    LLMSimple();
    ~LLMSimple();
    bool load_model(const String& path, int ctx_size = 8192);
    String generate(const String& prompt, int max_tokens = 256);
    void unload_model();
    bool is_model_loaded() const { return is_loaded; }
    String get_last_error() const { return last_error; }
    int get_context_size() const { return n_ctx; }
};
}"""

with open("llm_simple/include/llm_simple.h", "w") as f:
    f.write(header_content)
print(f"  ✓ {len(header_content)} caractères écrits")

# Implementation LLMSimple - MISE À JOUR POUR NOUVELLE API LLAMA.CPP
print("📄 Création llm_simple.cpp (nouvelle API llama.cpp)...")
impl_content = '''#include "llm_simple.h"
#include <godot_cpp/core/class_db.hpp>
#include <godot_cpp/classes/project_settings.hpp>
#include "llama.h"
#include <vector>

using namespace godot;

void LLMSimple::_bind_methods() {
    ClassDB::bind_method(D_METHOD("load_model","path","ctx_size"), &LLMSimple::load_model, DEFVAL(8192));
    ClassDB::bind_method(D_METHOD("generate","prompt","max_tokens"), &LLMSimple::generate, DEFVAL(256));
    ClassDB::bind_method(D_METHOD("unload_model"), &LLMSimple::unload_model);
    ClassDB::bind_method(D_METHOD("is_model_loaded"), &LLMSimple::is_model_loaded);
    ClassDB::bind_method(D_METHOD("get_last_error"), &LLMSimple::get_last_error);
    ClassDB::bind_method(D_METHOD("get_context_size"), &LLMSimple::get_context_size);
    ADD_SIGNAL(MethodInfo("generation_started"));
    ADD_SIGNAL(MethodInfo("generation_finished", PropertyInfo(Variant::STRING, "text")));
    ADD_SIGNAL(MethodInfo("generation_error", PropertyInfo(Variant::STRING, "error")));
}

LLMSimple::LLMSimple() {
    model = nullptr;
    ctx = nullptr;
    sampler = nullptr;
    n_ctx = 8192;
    is_loaded = false;
    llama_backend_init();
}

LLMSimple::~LLMSimple() {
    unload_model();
    llama_backend_free();
}

bool LLMSimple::load_model(const String& path, int ctx_size) {
    unload_model();
    n_ctx = ctx_size;

    String abs_path = ProjectSettings::get_singleton()->globalize_path(path);
    llama_model_params mp = llama_model_default_params();
    mp.n_gpu_layers = 0;

    model = llama_model_load_from_file(abs_path.utf8().get_data(), mp);
    if (!model) {
        last_error = "Failed to load model";
        return false;
    }

    llama_context_params cp = llama_context_default_params();
    cp.n_ctx = n_ctx;
    cp.n_batch = 512;
    cp.n_threads = 4;

    ctx = llama_init_from_model(model, cp);
    if (!ctx) {
        last_error = "Failed to create context";
        llama_model_free(model);
        model = nullptr;
        return false;
    }

    sampler = llama_sampler_chain_init(llama_sampler_chain_default_params());
    llama_sampler_chain_add(sampler, llama_sampler_init_temp(0.7f));
    llama_sampler_chain_add(sampler, llama_sampler_init_top_k(50));
    llama_sampler_chain_add(sampler, llama_sampler_init_top_p(0.95f, 1));
    llama_sampler_chain_add(sampler, llama_sampler_init_dist(1234));

    is_loaded = true;
    return true;
}

String LLMSimple::generate(const String& prompt, int max_tokens) {
    if (!is_loaded) return "";

    const llama_vocab * vocab = llama_model_get_vocab(model);

    std::vector<llama_token> tokens(n_ctx);
    int n = llama_tokenize(vocab, prompt.utf8().get_data(), prompt.length(),
                           tokens.data(), tokens.size(), true, false);
    if (n < 0) return "";

    tokens.resize(n);
    llama_sampler_reset(sampler);

    if (llama_decode(ctx, llama_batch_get_one(tokens.data(), n)) != 0) return "";

    String result;
    for (int i = 0; i < max_tokens; i++) {
        llama_token token = llama_sampler_sample(sampler, ctx, -1);
        if (llama_vocab_is_eog(vocab, token)) break;

        char buffer[256];
        int len = llama_token_to_piece(vocab, token, buffer, sizeof(buffer), 0, false);
        if (len < 0) break;

        result += String::utf8(buffer, len);
        if (llama_decode(ctx, llama_batch_get_one(&token, 1)) != 0) break;
    }

    return result;
}

void LLMSimple::unload_model() {
    if (sampler) { llama_sampler_free(sampler); sampler = nullptr; }
    if (ctx) { llama_free(ctx); ctx = nullptr; }
    if (model) { llama_model_free(model); model = nullptr; }
    is_loaded = false;
}'''

with open("llm_simple/src/llm_simple.cpp", "w") as f:
    f.write(impl_content)
print(f"  ✓ {len(impl_content)} caractères écrits")

# Registration
print("📄 Création register_types.cpp...")
register_content = '''#include "llm_simple.h"
#include <gdextension_interface.h>
#include <godot_cpp/godot.hpp>

using namespace godot;

void init(ModuleInitializationLevel level) {
    if (level != MODULE_INITIALIZATION_LEVEL_SCENE) return;
    ClassDB::register_class<LLMSimple>();
}

void uninit(ModuleInitializationLevel level) {}

extern "C" {
GDExtensionBool GDE_EXPORT llm_simple_library_init(
    GDExtensionInterfaceGetProcAddress p_get_proc_address,
    const GDExtensionClassLibraryPtr p_library,
    GDExtensionInitialization *r_initialization) {

    GDExtensionBinding::InitObject init_obj(p_get_proc_address, p_library, r_initialization);
    init_obj.register_initializer(init);
    init_obj.register_terminator(uninit);
    init_obj.set_minimum_library_initialization_level(MODULE_INITIALIZATION_LEVEL_SCENE);
    return init_obj.init();
}
}'''

with open("llm_simple/src/register_types.cpp", "w") as f:
    f.write(register_content)
print(f"  ✓ {len(register_content)} caractères écrits")

# GDExtension config
print("📄 Création llm_simple.gdextension...")
gdext_content = """[configuration]
entry_symbol = "llm_simple_library_init"
compatibility_minimum = "4.2"

[libraries]
windows.release.x86_64 = "res://addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll"
"""

with open("addons/llm_simple/llm_simple.gdextension", "w") as f:
    f.write(gdext_content)
print(f"  ✓ {len(gdext_content)} caractères écrits")

print()
print("🔍 Vérification fichiers créés:")
for file in ["llm_simple/include/llm_simple.h",
             "llm_simple/src/llm_simple.cpp",
             "llm_simple/src/register_types.cpp",
             "addons/llm_simple/llm_simple.gdextension"]:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"  ✓ {file} ({size} bytes)")
    else:
        raise Exception(f"❌ {file} non créé")

print()
print("="*70)
print("✅ CELLULE 2 TERMINÉE")
print("="*70)

📝 CELLULE 2: CRÉATION FICHIERS SOURCES

📄 Création llm_simple.h...
  ✓ 784 caractères écrits
📄 Création llm_simple.cpp (nouvelle API llama.cpp)...
  ✓ 3605 caractères écrits
📄 Création register_types.cpp...
  ✓ 836 caractères écrits
📄 Création llm_simple.gdextension...
  ✓ 202 caractères écrits

🔍 Vérification fichiers créés:
  ✓ llm_simple/include/llm_simple.h (784 bytes)
  ✓ llm_simple/src/llm_simple.cpp (3605 bytes)
  ✓ llm_simple/src/register_types.cpp (836 bytes)
  ✓ addons/llm_simple/llm_simple.gdextension (202 bytes)

✅ CELLULE 2 TERMINÉE


In [3]:
# ==========================================
# CELLULE 3: Téléchargement Dépendances
# ==========================================
import subprocess

print("="*70)
print("📥 CELLULE 3: TÉLÉCHARGEMENT DÉPENDANCES")
print("="*70)
print()

os.chdir("/content")

# Fonction helper
def git_clone(repo, branch=None, name=""):
    cmd = f"git clone --quiet --depth 1"
    if branch:
        cmd += f" --branch {branch}"
    cmd += f" {repo}"

    print(f"⏳ Clone {name}...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Erreur clone {name}")
        print("STDERR:", result.stderr)
        raise Exception(f"Clone failed: {name}")
    print(f"  ✓ {name} cloné")

# Godot-CPP
git_clone("https://github.com/godotengine/godot-cpp.git", "4.2", "godot-cpp")

# Vérification structure
if not os.path.exists("godot-cpp/gdextension/gdextension_interface.h"):
    raise Exception("❌ gdextension_interface.h non trouvé")
print("  ✓ gdextension_interface.h trouvé")

# Llama.cpp
git_clone("https://github.com/ggerganov/llama.cpp.git", None, "llama.cpp")

# Vérification structure llama.cpp
print()
print("🔍 Vérification structure llama.cpp:")
llama_files = [
    "ggml/src/ggml.c",
    "src/llama.cpp",
    "include/llama.h",
    "ggml/include/ggml.h"
]
for f in llama_files:
    path = f"llama.cpp/{f}"
    if os.path.exists(path):
        print(f"  ✓ {f}")
    else:
        print(f"  ❌ {f} MANQUANT")
        raise Exception(f"Fichier llama.cpp manquant: {f}")

print()
print("="*70)
print("✅ CELLULE 3 TERMINÉE")
print("="*70)

📥 CELLULE 3: TÉLÉCHARGEMENT DÉPENDANCES

⏳ Clone godot-cpp...
  ✓ godot-cpp cloné
  ✓ gdextension_interface.h trouvé
⏳ Clone llama.cpp...
  ✓ llama.cpp cloné

🔍 Vérification structure llama.cpp:
  ✓ ggml/src/ggml.c
  ✓ src/llama.cpp
  ✓ include/llama.h
  ✓ ggml/include/ggml.h

✅ CELLULE 3 TERMINÉE


In [4]:
# ==========================================
# CELLULE 4: Patch Godot-CPP pour MinGW
# ==========================================
print("="*70)
print("🔧 CELLULE 4: PATCH GODOT-CPP")
print("="*70)
print()

os.chdir("/content")

# Patch header
header_file = "godot-cpp/include/godot_cpp/core/class_db.hpp"
print(f"📝 Patch {header_file}...")

with open(header_file, "r") as f:
    content = f.read()

original_len = len(content)

# Remplacements
replacements = [
    ("static std::mutex engine_singletons_mutex;",
     "// static std::mutex engine_singletons_mutex;"),
    ("std::lock_guard<std::mutex> lock(engine_singletons_mutex);",
     "// std::lock_guard<std::mutex> lock(engine_singletons_mutex);")
]

for old, new in replacements:
    count = content.count(old)
    content = content.replace(old, new)
    print(f"  ✓ Remplacé '{old[:40]}...' ({count}x)")

with open(header_file, "w") as f:
    f.write(content)

print(f"  ✓ Header patché ({original_len} → {len(content)} bytes)")

# Patch source
source_file = "godot-cpp/src/core/class_db.cpp"
print(f"📝 Patch {source_file}...")

with open(source_file, "r") as f:
    content = f.read()

original_len = len(content)

replacements = [
    ("std::mutex ClassDB::engine_singletons_mutex;",
     "// std::mutex ClassDB::engine_singletons_mutex;"),
    ("std::lock_guard<std::mutex> lock(engine_singletons_mutex);",
     "// std::lock_guard<std::mutex> lock(engine_singletons_mutex);")
]

for old, new in replacements:
    count = content.count(old)
    content = content.replace(old, new)
    print(f"  ✓ Remplacé '{old[:40]}...' ({count}x)")

with open(source_file, "w") as f:
    f.write(content)

print(f"  ✓ Source patché ({original_len} → {len(content)} bytes)")

print()
print("="*70)
print("✅ CELLULE 4 TERMINÉE")
print("="*70)

🔧 CELLULE 4: PATCH GODOT-CPP

📝 Patch godot-cpp/include/godot_cpp/core/class_db.hpp...
  ✓ Remplacé 'static std::mutex engine_singletons_mute...' (1x)
  ✓ Remplacé 'std::lock_guard<std::mutex> lock(engine_...' (2x)
  ✓ Header patché (16192 → 16201 bytes)
📝 Patch godot-cpp/src/core/class_db.cpp...
  ✓ Remplacé 'std::mutex ClassDB::engine_singletons_mu...' (1x)
  ✓ Remplacé 'std::lock_guard<std::mutex> lock(engine_...' (1x)
  ✓ Source patché (19129 → 19135 bytes)

✅ CELLULE 4 TERMINÉE


In [5]:
# ==========================================
# CELLULE 5: Compilation Godot-CPP (~7 min)
# ==========================================
import time

print("="*70)
print("🔨 CELLULE 5: COMPILATION GODOT-CPP")
print("="*70)
print()

os.chdir("/content/godot-cpp")

print("⏳ Compilation en cours (7-8 minutes)...")
print("   Utilisation de tous les cœurs CPU disponibles")
print()

start_time = time.time()

result = subprocess.run(
    "scons platform=windows target=template_release arch=x86_64 use_mingw=yes -j$(nproc)",
    shell=True,
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

if result.returncode != 0:
    print("❌ ERREUR COMPILATION")
    print("\nDernières lignes stderr:")
    print(result.stderr[-2000:])
    print("\nDernières lignes stdout:")
    print(result.stdout[-2000:])
    raise Exception("Compilation godot-cpp échouée")

# Afficher fin de compilation
print("Dernières lignes:")
for line in result.stdout.split('\n')[-10:]:
    if line.strip():
        print(f"  {line}")

print()

# Vérification
lib_file = "bin/libgodot-cpp.windows.template_release.x86_64.a"
if not os.path.exists(lib_file):
    raise Exception(f"❌ {lib_file} non créé")

size_mb = os.path.getsize(lib_file) / (1024*1024)
print(f"✅ {lib_file}")
print(f"   Taille: {size_mb:.1f} MB")
print(f"   Durée: {elapsed/60:.1f} minutes")

print()
print("="*70)
print("✅ CELLULE 5 TERMINÉE")
print("="*70)

🔨 CELLULE 5: COMPILATION GODOT-CPP

⏳ Compilation en cours (7-8 minutes)...
   Utilisation de tous les cœurs CPU disponibles

Dernières lignes:
  Compiling gen/src/classes/margin_container.cpp ...
  Compiling gen/src/classes/path_follow2d.cpp ...
  Compiling gen/src/classes/capsule_shape2d.cpp ...
  Compiling gen/src/classes/rectangle_shape2d.cpp ...
  Compiling gen/src/classes/visual_shader_node_uv_func.cpp ...
  Compiling gen/src/classes/upnp_device.cpp ...
  Linking Static Library bin/libgodot-cpp.windows.template_release.x86_64.a ...
  Ranlib Library bin/libgodot-cpp.windows.template_release.x86_64.a ...
  scons: done building targets.

✅ bin/libgodot-cpp.windows.template_release.x86_64.a
   Taille: 69.3 MB
   Durée: 28.1 minutes

✅ CELLULE 5 TERMINÉE


In [6]:
# ==========================================
# CELLULE 6: Compilation Llama.cpp (~5 min)
# ==========================================
import glob

print("="*70)
print("🔨 CELLULE 6: COMPILATION LLAMA.CPP")
print("="*70)
print()

os.chdir("/content/llama.cpp")

# Macros requises - CORRIGÉ: ajout -fpermissive et -D_WIN32_WINNT pour compatibilité MinGW
DEFS = '-DGGML_VERSION=\\"1\\" -DGGML_COMMIT=\\"colab\\" -DGGML_BUILD_NUMBER=0'
CFLAGS = f"{DEFS} -DGGML_USE_CPU -O3 -DNDEBUG -fPIC -pthread"
CXXFLAGS = f"{DEFS} -O3 -DNDEBUG -std=c++17 -fPIC -pthread -fpermissive -D_WIN32_WINNT=0x0601 -Wno-attributes"

print("🔧 Compilation GGML (tous les fichiers source)...")
print(f"   Macros: {DEFS}")
print(f"   CXXFLAGS inclut: -fpermissive (fix std::async)")
print()

# Trouver TOUS les fichiers .c dans ggml/src
ggml_sources = glob.glob("ggml/src/*.c")
print(f"📋 Fichiers GGML trouvés: {len(ggml_sources)}")
for f in ggml_sources:
    print(f"   - {f}")
print()

# Compiler chaque fichier .c
ggml_objects = []
for src in ggml_sources:
    obj = os.path.basename(src).replace('.c', '.o')
    print(f"⏳ Compilation {src}...")

    cmd = f"x86_64-w64-mingw32-gcc -c {src} -Iggml/include -Iggml/src -Iinclude {CFLAGS}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ ERREUR compilation {src}")
        print("STDERR:", result.stderr[-1000:])
        raise Exception(f"Compilation {src} échouée")

    if os.path.exists(obj):
        ggml_objects.append(obj)
        print(f"   ✓ {obj}")
    else:
        print(f"   ⚠️ {obj} non créé (peut-être vide)")

print()
print(f"✅ {len(ggml_objects)} fichiers objets GGML compilés")

# Archivage GGML
if ggml_objects:
    obj_list = " ".join(ggml_objects)
    result = subprocess.run(
        f"x86_64-w64-mingw32-ar rcs /content/llama_build/libggml.a {obj_list}",
        shell=True, capture_output=True, text=True
    )

    if result.returncode != 0:
        raise Exception("❌ Archivage GGML échoué")

    # Nettoyer les .o
    for obj in ggml_objects:
        os.remove(obj)

size_kb = os.path.getsize("/content/llama_build/libggml.a") / 1024
print(f"✅ libggml.a ({size_kb:.0f} KB)")
print()

# Compilation LLAMA
print("🔧 Compilation LLAMA (tous les fichiers source)...")

# Trouver tous les fichiers .cpp dans src/
llama_sources = glob.glob("src/*.cpp")
print(f"📋 Fichiers LLAMA trouvés: {len(llama_sources)}")
for f in llama_sources:
    print(f"   - {f}")
print()

# Compiler chaque fichier .cpp
llama_objects = []
for src in llama_sources:
    obj = os.path.basename(src).replace('.cpp', '.o')
    print(f"⏳ Compilation {src}...")

    cmd = f"x86_64-w64-mingw32-g++ -c {src} -Iinclude -Iggml/include -Iggml/src -Icommon -Isrc {CXXFLAGS}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    # Afficher warnings mais continuer
    if "warning:" in result.stderr:
        warnings = len([l for l in result.stderr.split('\n') if 'warning:' in l])
        print(f"   ⚠️ {warnings} warnings")

    if result.returncode != 0:
        print(f"❌ ERREUR compilation {src}")
        print("STDERR:", result.stderr[-1500:])
        raise Exception(f"Compilation {src} échouée")

    if os.path.exists(obj):
        llama_objects.append(obj)
        print(f"   ✓ {obj}")
    else:
        print(f"   ⚠️ {obj} non créé")

print()
print(f"✅ {len(llama_objects)} fichiers objets LLAMA compilés")

# Archivage LLAMA
if llama_objects:
    obj_list = " ".join(llama_objects)
    result = subprocess.run(
        f"x86_64-w64-mingw32-ar rcs /content/llama_build/libllama.a {obj_list}",
        shell=True, capture_output=True, text=True
    )

    if result.returncode != 0:
        raise Exception("❌ Archivage LLAMA échoué")

    # Nettoyer les .o
    for obj in llama_objects:
        os.remove(obj)

size_kb = os.path.getsize("/content/llama_build/libllama.a") / 1024
print(f"✅ libllama.a ({size_kb:.0f} KB)")

print()
print("📊 Récapitulatif:")
os.system("ls -lh /content/llama_build/*.a")

print()
print("="*70)
print("✅ CELLULE 6 TERMINÉE")
print("="*70)

🔨 CELLULE 6: COMPILATION LLAMA.CPP

🔧 Compilation GGML (tous les fichiers source)...
   Macros: -DGGML_VERSION=\"1\" -DGGML_COMMIT=\"colab\" -DGGML_BUILD_NUMBER=0
   CXXFLAGS inclut: -fpermissive (fix std::async)

📋 Fichiers GGML trouvés: 3
   - ggml/src/ggml.c
   - ggml/src/ggml-alloc.c
   - ggml/src/ggml-quants.c

⏳ Compilation ggml/src/ggml.c...
   ✓ ggml.o
⏳ Compilation ggml/src/ggml-alloc.c...
   ✓ ggml-alloc.o
⏳ Compilation ggml/src/ggml-quants.c...
   ✓ ggml-quants.o

✅ 3 fichiers objets GGML compilés
✅ libggml.a (499 KB)

🔧 Compilation LLAMA (tous les fichiers source)...
📋 Fichiers LLAMA trouvés: 27
   - src/llama-memory-hybrid-iswa.cpp
   - src/llama-kv-cache-iswa.cpp
   - src/unicode-data.cpp
   - src/llama-arch.cpp
   - src/llama-sampling.cpp
   - src/llama-model.cpp
   - src/llama-cparams.cpp
   - src/llama-context.cpp
   - src/llama-vocab.cpp
   - src/llama-chat.cpp
   - src/llama-mmap.cpp
   - src/unicode.cpp
   - src/llama-kv-cache.cpp
   - src/llama-memory-recurrent.cpp

Exception: Compilation src/llama-model-loader.cpp échouée

In [ ]:
# ==========================================
# CELLULE 7: Compilation DLL Finale
# ==========================================
print("="*70)
print("🔨 CELLULE 7: COMPILATION DLL FINALE")
print("="*70)
print()

os.chdir("/content")

print("🔧 Linking final...")
print()

cmd = """x86_64-w64-mingw32-g++ -shared \\
    -o /content/addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll \\
    /content/llm_simple/src/llm_simple.cpp \\
    /content/llm_simple/src/register_types.cpp \\
    -I/content/llm_simple/include \\
    -I/content/godot-cpp/include \\
    -I/content/godot-cpp/gen/include \\
    -I/content/godot-cpp/gdextension \\
    -I/content/llama.cpp/include \\
    -I/content/llama.cpp/ggml/include \\
    -I/content/llama.cpp/common \\
    /content/godot-cpp/bin/libgodot-cpp.windows.template_release.x86_64.a \\
    /content/llama_build/libllama.a \\
    /content/llama_build/libggml.a \\
    -static-libgcc -static-libstdc++ -std=c++17 -O3 -pthread \\
    -Wl,--allow-multiple-definition"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ ERREUR LINKING")
    print("\n🔍 STDERR:")
    print(result.stderr[-2000:])
    print("\n🔍 STDOUT:")
    print(result.stdout[-1000:])
    raise Exception("Compilation DLL échouée")

dll_path = "/content/addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll"

if not os.path.exists(dll_path):
    raise Exception("❌ DLL non créée")

size_mb = os.path.getsize(dll_path) / (1024*1024)
print(f"✅ DLL créée!")
print(f"   Fichier: {dll_path}")
print(f"   Taille: {size_mb:.1f} MB")

print()
print("="*70)
print("✅ CELLULE 7 TERMINÉE")
print("="*70)

In [ ]:
# ==========================================
# CELLULE 8: Package et Téléchargement
# ==========================================
from google.colab import files

print("="*70)
print("📦 CELLULE 8: PACKAGE ET TÉLÉCHARGEMENT")
print("="*70)
print()

os.chdir("/content")

# README
print("📝 Création README...")
readme = """LLM SIMPLE pour Godot 4.2+

🎯 INSTALLATION:
1. Copier addons/llm_simple/ dans votre projet Godot
2. Télécharger le modèle Qwen2.5-3B-Instruct GGUF
3. Placer le fichier .gguf dans models/ de votre projet
4. Redémarrer Godot

🚀 USAGE RAPIDE:
extends Node

var llm: LLMSimple

func _ready():
    llm = LLMSimple.new()
    add_child(llm)

    if llm.load_model("res://models/qwen2.5-3b.gguf", 8192):
        var response = llm.generate("Bonjour! Qui es-tu?", 100)
        print(response)

func _exit_tree():
    if llm:
        llm.unload_model()

📥 MODÈLE RECOMMANDÉ:
Nom: Qwen2.5-3B-Instruct Q4_K_M
Taille: ~2 GB
URL: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF
Fichier: qwen2.5-3b-instruct-q4_k_m.gguf

✨ FONCTIONNALITÉS:
- load_model(path, context_size) : Charge un modèle GGUF
- generate(prompt, max_tokens) : Génère du texte
- unload_model() : Décharge le modèle
- is_model_loaded() : Vérifie si chargé
- get_last_error() : Récupère dernière erreur
- get_context_size() : Taille du contexte
"""

with open("addons/llm_simple/README.txt", "w") as f:
    f.write(readme)
print("  ✓ README.txt")

# Script de test
print("📝 Création TestLLM.gd...")
test_script = """extends Node

var llm: LLMSimple

func _ready():
    llm = LLMSimple.new()
    add_child(llm)

    print("🤖 Chargement du modèle Qwen2.5-3B...")

    if llm.load_model("res://models/qwen2.5-3b.gguf", 8192):
        print("✅ Modèle chargé! Context: ", llm.get_context_size())

        print("\\n📝 Test de génération...")
        var response = llm.generate("Bonjour! Qui es-tu?", 100)
        print("🎭 Réponse: ", response)
    else:
        print("❌ Erreur chargement: ", llm.get_last_error())
        print("⚠️ Vérifiez que le modèle existe dans models/")

func _exit_tree():
    if llm:
        llm.unload_model()
        print("👋 Modèle déchargé")
"""

with open("TestLLM.gd", "w") as f:
    f.write(test_script)
print("  ✓ TestLLM.gd")

# Création ZIP
print()
print("📦 Création du package ZIP...")
result = subprocess.run(
    "zip -q -r llm_simple_addon.zip addons/ TestLLM.gd",
    shell=True, capture_output=True
)

if result.returncode != 0:
    raise Exception("❌ Création ZIP échouée")

if not os.path.exists("llm_simple_addon.zip"):
    raise Exception("❌ ZIP non créé")

size_mb = os.path.getsize("llm_simple_addon.zip") / (1024*1024)
print(f"  ✓ llm_simple_addon.zip ({size_mb:.1f} MB)")

# Liste contenu
print()
print("📋 Contenu du package:")
os.system("unzip -l llm_simple_addon.zip | head -15")

# Téléchargement
print()
print("📥 Lancement du téléchargement...")
files.download("llm_simple_addon.zip")

print()
print("="*70)
print("🎉 COMPILATION RÉUSSIE!")
print("="*70)
print()
print("📋 PROCHAINES ÉTAPES:")
print()
print("1️⃣ Extraire llm_simple_addon.zip")
print("2️⃣ Copier addons/llm_simple/ dans votre projet Godot")
print("3️⃣ Télécharger Qwen2.5-3B-Instruct Q4:")
print("   https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF")
print("   Fichier: qwen2.5-3b-instruct-q4_k_m.gguf (~2 GB)")
print("4️⃣ Créer dossier models/ dans votre projet")
print("5️⃣ Renommer le fichier téléchargé en: qwen2.5-3b.gguf")
print("6️⃣ Placer dans: models/qwen2.5-3b.gguf")
print("7️⃣ Redémarrer Godot")
print("8️⃣ Attacher TestLLM.gd à un Node et tester!")
print()
print("="*70)
print("✅ CELLULE 8 TERMINÉE - TOUT EST PRÊT!")
print("="*70)